# 09 — Risk Score Construction (Injury Risk+)

## Goals
Combine multi-task model outputs into the final Injury Risk+ score. Calibrate,
validate, and normalize the score so it is interpretable and era-consistent.
Reconstruct historical scores for all pitchers from 2015–present.

**Full design specification:** `docs/injury_risk_plus_design.md`

## Inputs
- `data/processed/feature_matrix.parquet`
- `models/multitask_chained.joblib`
- `models/survival_rsf.pkl`

## Outputs
- `models/risk_plus_calibration.pkl`
- `models/risk_plus_normalization_reference.parquet`
- `data/processed/injury_risk_plus_scores.parquet` — historical scores, 2015–present
- `reports/figures/risk_plus_leaderboard.png`
- `reports/figures/risk_plus_time_series.png`

## Planned Analyses

### Score Construction
1. Calibrate injury probability predictions
2. Optimize blend weights (probability, severity, hazard)
3. Compute raw risk scores for all pitcher-game rows
4. Build normalization reference table by season × archetype
5. Normalize to Injury Risk+ scale (mean = 100)
6. Attach percentile ranks

### Validation
1. Does Injury Risk+ predict future injury better than each component alone?
2. Calibration: do pitchers with scores of 150 actually get injured 50% more than average?
3. Score stability: how much does a pitcher's score fluctuate game-to-game vs. season-to-season?
4. Historical leaderboard: which pitchers had the highest/lowest Injury Risk+ over their careers?

### Case Studies
- Select 3–5 well-known pitchers with injury histories and trace their Injury Risk+ trajectory
  leading up to each IL stint. Did the score spike before the injury?

## Future Work
- Body-region subscores (elbow-specific, shoulder-specific risk)
- Confidence intervals around each score using bootstrap resampling
- Career aggregate Injury Risk+ (cumulative risk over a pitcher's career)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.scoring.injury_risk_plus import (
    compute_raw_risk_score,
    normalize_to_injury_risk_plus,
    compute_seasonal_risk_plus,
    get_top_risk_pitchers,
    compute_risk_percentile,
)
from src.scoring.score_calibration import (
    calibrate_probabilities,
    optimize_blend_weights,
    build_normalization_reference,
)

# TODO: load models and feature matrix, construct scores